# Exercice 3 en C++ (notebook)
Méthode LR (factorisation LU) et méthode QR pour approcher les valeurs propres.

Ce notebook est prévu pour un noyau C++ (xeus-cling).

In [ ]:
#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>
#include <stdexcept>
#include <vector>

using Matrix = std::vector<std::vector<double>>;

Matrix zeros(int n, int m) {
    return Matrix(n, std::vector<double>(m, 0.0));
}

Matrix identity(int n) {
    Matrix I = zeros(n, n);
    for (int i = 0; i < n; ++i) I[i][i] = 1.0;
    return I;
}

Matrix transpose(const Matrix& A) {
    int n = static_cast<int>(A.size());
    int m = static_cast<int>(A[0].size());
    Matrix T = zeros(m, n);
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < m; ++j)
            T[j][i] = A[i][j];
    return T;
}

Matrix matmul(const Matrix& A, const Matrix& B) {
    int n = static_cast<int>(A.size());
    int p = static_cast<int>(A[0].size());
    int m = static_cast<int>(B[0].size());
    Matrix C = zeros(n, m);
    for (int i = 0; i < n; ++i) {
        for (int k = 0; k < p; ++k) {
            double aik = A[i][k];
            for (int j = 0; j < m; ++j) C[i][j] += aik * B[k][j];
        }
    }
    return C;
}

double off_diag_norm(const Matrix& A) {
    int n = static_cast<int>(A.size());
    double s = 0.0;
    for (int i = 0; i < n; ++i)
        for (int j = 0; j < n; ++j)
            if (i != j) s += A[i][j] * A[i][j];
    return std::sqrt(s);
}

std::vector<double> diag(const Matrix& A) {
    int n = static_cast<int>(A.size());
    std::vector<double> d(n);
    for (int i = 0; i < n; ++i) d[i] = A[i][i];
    return d;
}

void print_sorted(std::vector<double> v) {
    std::sort(v.begin(), v.end());
    std::cout << "[";
    for (size_t i = 0; i < v.size(); ++i) {
        std::cout << v[i] << (i + 1 < v.size() ? ", " : "");
    }
    std::cout << "]";
}

// LU de Doolittle sans pivotement (suffisant pour les matrices de l'exercice)
void lu_no_pivot(const Matrix& A, Matrix& L, Matrix& U) {
    int n = static_cast<int>(A.size());
    L = identity(n);
    U = zeros(n, n);

    for (int i = 0; i < n; ++i) {
        for (int k = i; k < n; ++k) {
            double sum = 0.0;
            for (int j = 0; j < i; ++j) sum += L[i][j] * U[j][k];
            U[i][k] = A[i][k] - sum;
        }

        if (std::abs(U[i][i]) < 1e-14) {
            throw std::runtime_error("Pivot nul: LU sans pivotement impossible.");
        }

        for (int k = i + 1; k < n; ++k) {
            double sum = 0.0;
            for (int j = 0; j < i; ++j) sum += L[k][j] * U[j][i];
            L[k][i] = (A[k][i] - sum) / U[i][i];
        }
    }
}

// QR par Gram-Schmidt classique
void qr_gs(const Matrix& A, Matrix& Q, Matrix& R) {
    int n = static_cast<int>(A.size());
    Q = zeros(n, n);
    R = zeros(n, n);

    std::vector<std::vector<double>> V = A;

    for (int j = 0; j < n; ++j) {
        for (int i = 0; i < j; ++i) {
            double rij = 0.0;
            for (int k = 0; k < n; ++k) rij += Q[k][i] * V[k][j];
            R[i][j] = rij;
            for (int k = 0; k < n; ++k) V[k][j] -= rij * Q[k][i];
        }

        double norm = 0.0;
        for (int k = 0; k < n; ++k) norm += V[k][j] * V[k][j];
        norm = std::sqrt(norm);
        R[j][j] = norm;

        if (norm < 1e-14) throw std::runtime_error("Colonne quasi nulle pendant QR.");
        for (int k = 0; k < n; ++k) Q[k][j] = V[k][j] / norm;
    }
}

Matrix lr_iter(Matrix A, int n_iter) {
    for (int it = 0; it < n_iter; ++it) {
        Matrix L, U;
        lu_no_pivot(A, L, U);
        A = matmul(U, L);
    }
    return A;
}

Matrix qr_iter(Matrix A, int n_iter) {
    for (int it = 0; it < n_iter; ++it) {
        Matrix Q, R;
        qr_gs(A, Q, R);
        A = matmul(R, Q);
    }
    return A;
}

int main() {
    std::cout << std::fixed << std::setprecision(8);

    Matrix A = {
        {2.0, -1.0, 0.0},
        {-1.0, 2.0, -1.0},
        {0.0, -1.0, 2.0}
    };

    Matrix B = {
        {1.0 / std::sqrt(2.0), 1.0 / std::sqrt(2.0)},
        {1.0 / std::sqrt(2.0), -1.0 / std::sqrt(2.0)}
    };

    const int n_iter = 30;

    std::cout << "===== Matrice A =====\n";
    std::cout << "Valeurs propres exactes (theorie): [0.58578644, 2.00000000, 3.41421356]\n";

    Matrix A_lr = lr_iter(A, n_iter);
    Matrix A_qr = qr_iter(A, n_iter);

    std::cout << "[LR / LU] diag approx = ";
    print_sorted(diag(A_lr));
    std::cout << ", norme hors diagonale = " << off_diag_norm(A_lr) << "\n";

    std::cout << "[QR]      diag approx = ";
    print_sorted(diag(A_qr));
    std::cout << ", norme hors diagonale = " << off_diag_norm(A_qr) << "\n\n";

    std::cout << "===== Matrice B (orthogonale) =====\n";
    std::cout << "Valeurs propres exactes (theorie): [-1.00000000, 1.00000000]\n";

    Matrix B_lr = lr_iter(B, n_iter);
    Matrix B_qr = qr_iter(B, n_iter);

    std::cout << "[LR / LU] diag approx = ";
    print_sorted(diag(B_lr));
    std::cout << ", norme hors diagonale = " << off_diag_norm(B_lr) << "\n";

    std::cout << "[QR]      diag approx = ";
    print_sorted(diag(B_qr));
    std::cout << ", norme hors diagonale = " << off_diag_norm(B_qr) << "\n\n";

    std::cout << "Constat: QR est en general plus robuste et plus stable numeriquement que LR/LU.\n";
    return 0;
}

main();